In [0]:
df = spark.read.format("csv").option("header", "true").load("/Volumes/external-catalog/default/volume_first/Employee_Attrition.csv")
display(df)

In [0]:
# Filter for high risk attrition employees
high_risk_df = df.filter((df.Attrition == "No") & (df.JobSatisfaction.cast("int") < 3))

# Select relevant columns
selected_df = high_risk_df.select("EmployeeNumber", "Attrition", "JobSatisfaction", "Department", "JobRole", "Age")

# Write to Delta table in default schema of external-catalog
selected_df.write.format("delta").mode("overwrite").saveAsTable("`external-catalog`.default.high_risk_attrition_employees")

In [0]:
history_df = spark.sql("DESCRIBE HISTORY `external-catalog`.default.high_risk_attrition_employees")
display(history_df.select("version", "timestamp", "operation"))

In [0]:
df_delta = spark.read.format("delta").table("`external-catalog`.default.high_risk_attrition_employees")
display(df_delta)

In [0]:
spark.sql("""
INSERT INTO `external-catalog`.default.high_risk_attrition_employees (EmployeeNumber, Attrition, JobSatisfaction, Department, JobRole, Age)
VALUES (99999, 'No', 1, 'DummyDept', 'DummyRole', 30)
""")

In [0]:
df_delta = spark.read.format("delta").table("`external-catalog`.default.high_risk_attrition_employees")
display(df_delta)

In [0]:
df_delta_v0 = spark.read.option("versionAsOf", 0).table("`external-catalog`.default.high_risk_attrition_employees")
display(df_delta_v0)

In [0]:
df_delta_ts = spark.read.option("timestampAsOf", "2026-01-08T20:24:14.192+00:00").table("`external-catalog`.default.high_risk_attrition_employees")
display(df_delta_ts)

In [0]:
df_before_ts = spark.read.option("timestampAsOf", "2026-01-08T20:24:14.191+00:00").table("`external-catalog`.default.high_risk_attrition_employees")
display(df_before_ts)

In [0]:
spark.sql("CREATE VOLUME `external-catalog`.default.emplyee_transformed_data")

In [0]:
# Simple logical transformation: Add a column 'IsYoung' indicating if Age < 30
emplyee_df = spark.read.table("`external-catalog`.default.employee_attrition")
transformed_df = emplyee_df.withColumn("IsYoung", emplyee_df.Age.cast("int") < 30)

# Save to the newly created volume as a Delta table
transformed_df.write.format("delta").mode("overwrite").save("/Volumes/external-catalog/default/emplyee_transformed_data")

In [0]:
display(df_before_ts)